# Peak Current and Passives for the 120V to 12V Buck

Created with Grok Build

This is the wxMaxima 13.04.2 worksheet linked from the [120 V to 12 V buck](buck_120V_to_12V_100mA.md), evaluated in Python. The input in the sheet is 130 V, the maximum in the requirements, and the output is 12 V. Each code cell is the worksheet expression. The printed line is that expression's value.


## Peak current resistor selection

`I_peak` is 0.1 A. `R_iset` is that current times 10^6, in ohms. The input is 130 V and the output is 12 V.

The note says the frequency will be optimized iteratively, starting from a 100 kHz estimate. The cell that is saved sets the frequency to 200 kHz.

$$
L = \frac{V_{out}}{f \, I_{peak}} \left(1 - \frac{V_{out}}{V_{in}}\right)
$$

The minimum on time is taken from page 11 of the datasheet as 150 ns. Two checks ask whether the inductance is above the minimum. The inductor actually selected is 500 µH.


In [1]:
import math

def show(label, value, unit):
    print(f"{label:<18} {value:.6g} {unit}")

I_peak = 0.1
R_iset = I_peak * 10**6
V_in = 130
V_out = 12
freq = 200_000
L = (V_out / (freq * I_peak)) * (1 - V_out / V_in)
t_on_min = 150e-9
L_test1 = ((V_in * t_on_min) / I_peak) * 1.2
L_test2 = ((V_out * 3.5e-6) / 1) * 1.2
L_actual = 500e-6

show("I_peak", I_peak, "A")
show("R_iset", R_iset / 1e3, "kohm")
show("V_in", V_in, "V")
show("V_out", V_out, "V")
show("freq", freq, "Hz")
show("L", L * 1e6, "µH")
show("t_on_min", t_on_min * 1e9, "ns")
show("L_test1", L_test1 * 1e6, "µH")
show("L_test2", L_test2 * 1e6, "µH")
show("L_actual", L_actual * 1e6, "µH")


I_peak             0.1 A
R_iset             100 kohm
V_in               130 V
V_out              12 V
freq               200000 Hz
L                  544.615 µH
t_on_min           150 ns
L_test1            234 µH
L_test2            50.4 µH
L_actual           500 µH


## Input and output capacitors

The allowable input ripple is 25 mV. The source is taken to be stiff. The minimum input capacitance is

$$
C_{in} = \frac{L \, I_{peak}^{2}}{2 \, V_{in} \, \Delta V_{in}}
$$

A 1 µF ceramic is used, for the margin and for low ESR. The RMS ripple current in that capacitor is

$$
I_{RMS,in} = I_{peak} \frac{V_{out}}{V_{in}} \sqrt{\frac{V_{in}}{V_{out}} - 1}
$$

The acceptable output ripple is 50 mV. The sheet's minimum output capacitance is

$$
C_{out,min} = 100 \, \frac{L_{actual}}{2} \left(\frac{I_{peak}}{V_{out}}\right)^{2}
$$

The worst-case output-capacitor ripple current is half of `I_peak`. With an ESR of 0.3 ohm, the dissipation is the square of that current times the ESR.


In [2]:
delta_V_in = 25e-3
Cin = (L * I_peak**2) / (2 * V_in * delta_V_in)
C_in_actual = 1e-6
I_RMS_in = I_peak * (V_out / V_in) * math.sqrt((V_in / V_out) - 1)
delta_V_out = 50e-3
C_out_min = ((L_actual / 2) * (I_peak / V_out) ** 2) * 100
I_out_ripple_WC = I_peak / 2
ESR_C_out = 0.3
P_diss_C_out = I_out_ripple_WC**2 * ESR_C_out

show("delta_V_in", delta_V_in * 1e3, "mV")
show("Cin", Cin * 1e6, "µF")
show("C_in_actual", C_in_actual * 1e6, "µF")
show("I_RMS_in", I_RMS_in * 1e3, "mA")
show("delta_V_out", delta_V_out * 1e3, "mV")
show("C_out_min", C_out_min * 1e6, "µF")
show("I_out_ripple_WC", I_out_ripple_WC * 1e3, "mA")
show("ESR_C_out", ESR_C_out, "ohm")
show("P_diss_C_out", P_diss_C_out * 1e3, "mW")


delta_V_in         25 mV
Cin                0.83787 µF
C_in_actual        1 µF
I_RMS_in           28.946 mA
delta_V_out        50 mV
C_out_min          1.73611 µF
I_out_ripple_WC    50 mA
ESR_C_out          0.3 ohm
P_diss_C_out       0.75 mW


## Input filter

The snubber capacitance is four times the 1 µF input capacitor. The input-filter inductor is picked at 100 µH, to be refined later. The snubber resistor is

$$
R_{snubber} = \sqrt{\frac{L_{filter}}{C_{snubber}}}
$$


In [3]:
C_in_snubber = 4 * C_in_actual
L_in_filter = 100e-6
R_snubber = math.sqrt(L_in_filter / C_in_snubber)

show("C_in_snubber", C_in_snubber * 1e6, "µF")
show("L_in_filter", L_in_filter * 1e6, "µH")
show("R_snubber", R_snubber, "ohm")


C_in_snubber       4 µF
L_in_filter        100 µH
R_snubber          5 ohm


## Feedback divider

The first trial fixes the lower resistor at 100 kohm and solves

$$
V_{out} = 0.8 \left(1 + \frac{R_1}{R_2}\right)
$$

for the upper resistor. That upper resistor is 1.4 Mohm. The sheet calls it impractical, because the circuit is less stable at low load current with it.

The sheet then sets the voltage the IC regulates to 5 V and divides the 12 V output down to 5 V, using Figure 8 on page 14, with the lower resistor at 10 kohm. The divider equation in the sheet is

$$
V_b = V_a \frac{R_2}{R_1 + R_2}
$$

which solves to $R_1 = R_2 (V_a - V_b) / V_b$. The numeric cell that follows is entered as `-(V_midpoint - V_out * R2_fb) / V_midpoint`, with the midpoint defined as 5 V.


In [4]:
R2_fb_trial = 100e3
R1_fb_trial = R2_fb_trial * (V_out / 0.8 - 1)

R2_fb = 10e3
# Closed form of solve(Vb = Va * R2 / (R1 + R2), R1).
# Va is the 12 V output and Vb is the 5 V midpoint.
R1_from_divider = R2_fb * (V_out - 5) / 5

V_midpoint = 5
R1_fb = -(V_midpoint - V_out * R2_fb) / V_midpoint

show("R2 trial", R2_fb_trial / 1e3, "kohm")
show("R1 trial", R1_fb_trial / 1e6, "Mohm")
show("R2", R2_fb / 1e3, "kohm")
show("R1 from divider", R1_from_divider / 1e3, "kohm")
show("R1 as entered", R1_fb / 1e3, "kohm")


R2 trial           100 kohm
R1 trial           1.4 Mohm
R2                 10 kohm
R1 from divider    14 kohm
R1 as entered      23.999 kohm


## Undervoltage and overvoltage lockout

The lockout string is 2 Mohm total. The overvoltage threshold is 135 V and the undervoltage threshold is 115 V. R5, R4, and R3 follow Figure 10 on page 15 of the datasheet:

$$
R_5 = R_{total} \frac{1.21}{V_{OVLO}}, \quad
R_4 = R_{total} \frac{1.21}{V_{UVLO}} - R_5, \quad
R_3 = R_{total} - R_5 - R_4
$$

The values actually used are 18 kohm, 3.16 kohm, and 1.98 Mohm. The sheet then solves those resistors back to the two thresholds. Page 15 requires the OVLO pin to stay under 6 V. The pin voltage is the 130 V input times R5 over the actual string.


In [5]:
R_lockout_total = 2e6
OVLO_threshold = 135
UVLO_threshold = 115
R5 = R_lockout_total * (1.21 / OVLO_threshold)
R4 = (R_lockout_total * (1.21 / UVLO_threshold)) - R5
R3 = R_lockout_total - R5 - R4

R5_actual = 18e3
R4_actual = 3.16e3
R3_actual = 1.98e6
R_total_actual = R5_actual + R4_actual + R3_actual
OVLO_actual = (121 * R_total_actual) / (100 * R5_actual)
UVLO_actual = (121 * R_total_actual) / (100 * R5_actual + 100 * R4_actual)
V_ovlo_pin = V_in * (R5_actual / R_total_actual)

show("R5 calculated", R5 / 1e3, "kohm")
show("R4 calculated", R4 / 1e3, "kohm")
show("R3 calculated", R3 / 1e6, "Mohm")
show("R5 actual", R5_actual / 1e3, "kohm")
show("R4 actual", R4_actual / 1e3, "kohm")
show("R3 actual", R3_actual / 1e6, "Mohm")
show("R total actual", R_total_actual / 1e6, "Mohm")
show("OVLO actual", OVLO_actual, "V")
show("UVLO actual", UVLO_actual, "V")
show("OVLO pin", V_ovlo_pin, "V")


R5 calculated      17.9259 kohm
R4 calculated      3.11755 kohm
R3 calculated      1.97896 Mohm
R5 actual          18 kohm
R4 actual          3.16 kohm
R3 actual          1.98 Mohm
R total actual     2.00116 Mohm
OVLO actual        134.522 V
UVLO actual        114.433 V
OVLO pin           1.16932 V


## Soft-start capacitor

The desired soft-start delay is 100 ms. The capacitor is

$$
C_{SS} = t_{SS} \, \frac{5 \times 10^{-6}}{0.8}
$$

The standard value chosen is 680 nF. The soft-start time with that capacitor is

$$
t_{SS} = \frac{C_{SS} \, 0.8}{5 \times 10^{-6}}
$$


In [6]:
SS_time = 100e-3
C_SS = SS_time * ((5e-6) / 0.8)
C_SS_actual = 680e-9
SS_time_actual = (C_SS_actual * 0.8) / (5e-6)

show("SS_time desired", SS_time * 1e3, "ms")
show("C_SS", C_SS * 1e9, "nF")
show("C_SS actual", C_SS_actual * 1e9, "nF")
show("SS_time actual", SS_time_actual * 1e3, "ms")


SS_time desired    100 ms
C_SS               625 nF
C_SS actual        680 nF
SS_time actual     108.8 ms


## Parts taken from the sheet

The cells above select 500 µH for the power inductor, 1 µF for the input capacitor, 100 µH for the input filter, 5 ohm for the snubber resistor, 18 kohm / 3.16 kohm / 1.98 Mohm for the lockout string, and 680 nF for soft start.
